# Data analysis

Business case: Scouting undervalued football players to strenghten weaker underperforming players

In [2]:
-- =====================================================
-- QUERY 1A: USG FLANK REPLACEMENT CANDIDATES
-- Rule: Age > 30 OR performance >15% below team average
-- Tables: player, player_stat
-- =====================================================
 
WITH usg_flank_base AS (
    -- Pull all USG flank players for current season
    SELECT
        p.PlayerID,
        p.Player,
        p.Age,
        p.Pos,
        p.MP,
        p.Min,
        p."90s",
        p.Gls,
        p.Ast,
        p.xAG,
        p.PrgC,
        p.PrgR
    FROM player p
    WHERE p.TeamID = 'e14f61a5'       -- Union SG TeamID
      AND p.Season  = '2025-2026'
      AND (p.Pos LIKE '%FW%' OR p.Pos LIKE '%MF%')
      AND p.Min > 300                  -- Min. minutes threshold for significance
),
 
team_avg AS (
    -- Compute team-level per-90 benchmarks for all USG flanks
    SELECT
        AVG(p.xAG  / NULLIF(p."90s", 0)) AS avg_xAG_p90,
        AVG(p.PrgC / NULLIF(p."90s", 0)) AS avg_PrgC_p90,
        AVG(p.PrgR / NULLIF(p."90s", 0)) AS avg_PrgR_p90,
        AVG(p.Gls  / NULLIF(p."90s", 0)) AS avg_Gls_p90,
        AVG(p.Ast  / NULLIF(p."90s", 0)) AS avg_Ast_p90
    FROM player p
    WHERE p.TeamID = 'e14f61a5'
      AND p.Season  = '2025-2026'
      AND (p.Pos LIKE '%FW%' OR p.Pos LIKE '%MF%')
      AND p.Min > 300
)
 
SELECT
    f.Player,
    f.Age,
    f.Pos,
    f.MP,
    ROUND(f.Min, 0)                                         AS Minutes,
    ROUND(f.xAG  / NULLIF(f."90s", 0), 3)                  AS xAG_p90,
    ROUND(f.PrgC / NULLIF(f."90s", 0), 2)                  AS PrgC_p90,
    ROUND(f.PrgR / NULLIF(f."90s", 0), 2)                  AS PrgR_p90,
    t.avg_xAG_p90,
    t.avg_PrgC_p90,
    -- Flag: Age >30 OR any key metric >15% below team average
    CASE
        WHEN CAST(SUBSTR(f.Age, 1, 2) AS INTEGER) > 30 THEN 'AGE_RISK'
        WHEN (f.xAG  / NULLIF(f."90s", 0)) < (t.avg_xAG_p90  * 0.85) THEN 'PERF_STAGNATION'
        WHEN (f.PrgC / NULLIF(f."90s", 0)) < (t.avg_PrgC_p90 * 0.85) THEN 'PERF_STAGNATION'
        ELSE 'ACCEPTABLE'
    END                                                     AS replacement_flag
FROM usg_flank_base f
CROSS JOIN team_avg t
ORDER BY replacement_flag DESC, f.Age DESC;


SyntaxError: invalid decimal literal (2445135174.py, line 2)

In [4]:
-- =====================================================
-- QUERY 1B: SCOUTING TARGETS — YOUNG HIGH-POTENTIAL WINGERS
-- Rule: Age 18-24, High Progressive Carries, High xA
-- Tables: tmPlayer (Transfermarkt), player, player_stat
-- =====================================================
 
WITH prospect_stats AS (
    -- Aggregate per-season match-level stats for young Belgian Pro League wingers
    SELECT
        ps.PlayerID,
        SUM(ps.Min)                                          AS total_min,
        SUM(ps.Min) / 90.0                                   AS nineties,
        SUM(ps.Carries_PrgC)                                 AS total_PrgC,
        SUM(ps.Expected_xAG)                                 AS total_xAG,
        SUM(ps."Take-Ons_Succ")                              AS total_dribbles,
        SUM(ps.Performance_Crs)                              AS total_crosses,
        SUM(ps.Performance_TklW)                             AS total_tkl_wins,
        SUM(ps.Performance_Int)                              AS total_interceptions
    FROM player_stat ps
    WHERE ps.Min > 0
    GROUP BY ps.PlayerID
),
 
young_wingers AS (
    SELECT
        p.PlayerID,
        p.Player,
        p.Age,
        p.Pos,
        p.Season,
        tm.Position   AS tm_position,
        tm.Value      AS market_value,
        tm.Team       AS current_club,
        -- Per-90 metrics
        ROUND(ps.total_xAG   / NULLIF(ps.nineties, 0), 3)  AS xAG_p90,
        ROUND(ps.total_PrgC  / NULLIF(ps.nineties, 0), 2)  AS PrgC_p90,
        ROUND(ps.total_dribbles / NULLIF(ps.nineties, 0), 2) AS Dribbles_p90,
        ROUND(ps.total_crosses  / NULLIF(ps.nineties, 0), 2) AS Crosses_p90,
        ROUND((ps.total_tkl_wins + ps.total_interceptions) / NULLIF(ps.nineties, 0), 2) AS DefRec_p90,
        ps.total_min
    FROM player p
    JOIN prospect_stats ps ON p.PlayerID = ps.PlayerID
    JOIN tmPlayer tm ON p.Player = tm.Name
    WHERE p.Season = '2024-2025'
      AND CAST(SUBSTR(p.Age, 1, 2) AS INTEGER) BETWEEN 18 AND 24
      AND (
              tm.Position IN ('Left Winger', 'Right Winger',
                              'Left Midfield', 'Right Midfield',
                              'Attacking Midfield')
          )
      AND ps.total_min >= 450          -- Meaningful sample (5 full games)
)
 
SELECT *
FROM young_wingers
WHERE xAG_p90  >= (SELECT PERCENTILE_CONT(0.6) WITHIN GROUP (ORDER BY xAG_p90)  FROM young_wingers)
  AND PrgC_p90 >= (SELECT PERCENTILE_CONT(0.6) WITHIN GROUP (ORDER BY PrgC_p90) FROM young_wingers)
ORDER BY xAG_p90 DESC, PrgC_p90 DESC
LIMIT 20;
"""
 
 


SyntaxError: invalid decimal literal (1999268458.py, line 2)